# AI Spot Trading — Dataset Builder

Builds the training/validation/agent arrays once and publishes them as a Kaggle Dataset.
Both the **Colab** and **Kaggle** training notebooks then import this dataset instead of
downloading + rebuilding features every run.

## How to publish
1. Run this notebook end-to-end on **Kaggle** (no GPU required).
2. After it finishes, `dataset.npz` lands in `/kaggle/working/`.
3. From the notebook **Output**, "Create new dataset" with slug `acaurangel/ai-spot-trading-dataset`.
4. New versions: run again, then publish a **new version** of the same dataset.

In [ ]:
!pip install -q ccxt PyWavelets

In [ ]:
import math, time, os, json
import numpy as np
import ccxt

# Dataset-relevant hyperparameters (MUST match the training notebooks)
WINDOW_SIZE          = 128            # 128 x 15min ~ 32h of context
HORIZON              = 4              # predict direction 4 x 15min = 60 min ahead
NUM_FEATURES         = 16             # 10 original + 6 wavelet

# Constants
INDICATOR_WARMUP      = 200
RETURN_CLIP           = 0.1
VOLATILITY_WINDOW     = 20
BACKTEST_HOLDOUT      = 1000
VALIDATION_FRACTION   = 0.1
EMBARGO_FRACTION      = 0.01
HISTORICAL_CANDLES    = 35_000        # 35k x 15min ~ 365 days

STOP_LOSS_PCT    = 0.0030
TAKE_PROFIT_PCT  = 0.0050
SLIPPAGE_PCT     = 0.0005

SOFT_LABEL_SCALE     = 400            # MUST match the training notebooks
SOFT_LABEL_CLIP      = (0.02, 0.98)

FORECASTER_SYMBOLS = [
    'BTC/USDT', 'ETH/USDT',
    'SOL/USDT', 'LINK/USDT', 'BNB/USDT',
    'XRP/USDT', 'DOGE/USDT'
]
AGENT_SYMBOL = 'BTC/USDT'
TIMEFRAME    = '15m'

if os.path.isdir('/kaggle/working'):
    WORK_DIR = '/kaggle/working'
elif os.path.isdir('/content'):
    WORK_DIR = '/content'
else:
    WORK_DIR = '.'
print(f'WORK_DIR = {WORK_DIR}')

In [ ]:
# ── Technical indicators (pure NumPy) ────────────────────────────────────────

def ema(values: np.ndarray, period: int) -> np.ndarray:
    alpha = 2.0 / (period + 1)
    result = np.full(len(values), np.nan)
    if len(values) < period:
        return result
    result[period - 1] = np.mean(values[:period])
    for i in range(period, len(values)):
        result[i] = values[i] * alpha + result[i - 1] * (1 - alpha)
    return result

def rsi(values: np.ndarray, period: int = 14) -> np.ndarray:
    deltas = np.diff(values)
    result = np.full(len(values), np.nan)
    if len(deltas) < period:
        return result
    gains = np.where(deltas > 0, deltas, 0.0)
    losses = np.where(deltas < 0, -deltas, 0.0)
    avg_g, avg_l = np.mean(gains[:period]), np.mean(losses[:period])
    rs = avg_g / (avg_l + 1e-10)
    result[period] = 100 - 100 / (1 + rs)
    for i in range(period, len(deltas)):
        avg_g = (avg_g * (period - 1) + gains[i]) / period
        avg_l = (avg_l * (period - 1) + losses[i]) / period
        rs = avg_g / (avg_l + 1e-10)
        result[i + 1] = 100 - 100 / (1 + rs)
    return result

def macd(values: np.ndarray, fast=12, slow=26, signal=9):
    fast_ema = ema(values, fast)
    slow_ema = ema(values, slow)
    macd_line = fast_ema - slow_ema
    signal_line = ema(np.where(np.isnan(macd_line), 0, macd_line), signal)
    return macd_line, signal_line

def bollinger(values: np.ndarray, period: int = 20, std_dev: float = 2.0):
    upper = np.full(len(values), np.nan)
    lower = np.full(len(values), np.nan)
    for i in range(period - 1, len(values)):
        window = values[i - period + 1 : i + 1]
        m = np.mean(window)
        s = np.std(window, ddof=0)
        upper[i] = m + std_dev * s
        lower[i] = m - std_dev * s
    return upper, lower

print("Indicators OK.")

# ── Wavelet decomposition (dual-path frequency features) ─────────────────────
# Decomposes close prices into frequency bands using Discrete Wavelet Transform.
# Level-2 DWT produces: cA2 (trend), cD2 (medium-freq), cD1 (high-freq/noise).
# This separates the signal into components that capture:
#   - cA2: underlying trend (denoised price direction)
#   - cD2: medium-frequency cycles (multi-bar momentum patterns)
#   - cD1: high-frequency noise (tick-level volatility, market microstructure)
#
# We use Daubechies-4 wavelet ('db4') which is standard for financial signals.

import pywt
import numpy as np

WAVELET_NAME  = 'db4'
WAVELET_LEVEL = 2

def wavelet_decompose_series(closes: np.ndarray) -> dict:
    """Decompose full close series into frequency bands.
    Returns dict with arrays aligned to original length."""
    n = len(closes)

    # Pad to nearest power of 2 for clean DWT
    pad_len = int(2 ** np.ceil(np.log2(n)))
    padded = np.pad(closes, (0, pad_len - n), mode='edge')

    # Multilevel DWT: returns [cA2, cD2, cD1]
    coeffs = pywt.wavedec(padded, WAVELET_NAME, level=WAVELET_LEVEL)

    # Reconstruct each band at full resolution
    # Trend (low-freq): zero out detail coefficients
    trend_coeffs = [coeffs[0]] + [np.zeros_like(c) for c in coeffs[1:]]
    trend = pywt.waverec(trend_coeffs, WAVELET_NAME)[:n]

    # Medium-freq: zero out approximation and high-freq detail
    med_coeffs = [np.zeros_like(coeffs[0]), coeffs[1]] + \
                 [np.zeros_like(c) for c in coeffs[2:]]
    medium = pywt.waverec(med_coeffs, WAVELET_NAME)[:n]

    # High-freq: zero out everything except finest detail
    high_coeffs = [np.zeros_like(coeffs[0])] + \
                  [np.zeros_like(c) for c in coeffs[1:-1]] + [coeffs[-1]]
    high = pywt.waverec(high_coeffs, WAVELET_NAME)[:n]

    return {'trend': trend, 'medium': medium, 'high': high}


def wavelet_energy_ratio(detail: np.ndarray, window: int = 20) -> np.ndarray:
    """Rolling energy (squared magnitude) of a wavelet band."""
    energy = np.zeros(len(detail))
    for i in range(window, len(detail)):
        segment = detail[i - window : i]
        energy[i] = np.mean(segment ** 2)
    return energy


def wavelet_trend_slope(trend: np.ndarray, window: int = 10) -> np.ndarray:
    """Rolling slope of the wavelet trend (direction + strength)."""
    slope = np.zeros(len(trend))
    for i in range(window, len(trend)):
        segment = trend[i - window : i]
        x = np.arange(window)
        slope[i] = np.polyfit(x, segment, 1)[0]
    return slope


def spectral_entropy_rolling(closes: np.ndarray, window: int = 32) -> np.ndarray:
    """Rolling spectral entropy — measures how 'complex' the frequency content is.
    High entropy = noisy/chaotic, low entropy = dominated by few frequencies."""
    entropy = np.zeros(len(closes))
    for i in range(window, len(closes)):
        segment = closes[i - window : i]
        fft_amp = np.abs(np.fft.rfft(segment - np.mean(segment)))
        psd = fft_amp ** 2
        psd_norm = psd / (np.sum(psd) + 1e-12)
        entropy[i] = -np.sum(psd_norm * np.log2(psd_norm + 1e-12))
    # Normalize to [0, 1]
    max_ent = np.log2(window // 2 + 1)  # max possible entropy
    entropy = entropy / max_ent
    return entropy

print("Wavelet decomposition + spectral entropy OK.")


In [ ]:
# ── Candle fetch — multi-exchange with fallback ───────────────────────────────

TIMEFRAME_MS = {
    "1m": 60_000, "3m": 180_000, "5m": 300_000, "15m": 900_000,
    "30m": 1_800_000, "1h": 3_600_000, "2h": 7_200_000, "4h": 14_400_000,
    "6h": 21_600_000, "8h": 28_800_000, "12h": 43_200_000, "1d": 86_400_000,
}
MAX_PER_REQUEST = 1000

EXCHANGE_CONFIGS = [
    (lambda: ccxt.bybit({"enableRateLimit": True}),  {"category": "spot"}),
    (lambda: ccxt.okx({"enableRateLimit": True}),    {}),
    (lambda: ccxt.kucoin({"enableRateLimit": True}), {}),
    (lambda: ccxt.gate({"enableRateLimit": True}),   {}),
    (lambda: ccxt.mexc({"enableRateLimit": True}),   {}),
]

_active_exchange = None
_active_params   = {}

def _get_exchange():
    global _active_exchange, _active_params
    for factory, extra_params in EXCHANGE_CONFIGS:
        ex = factory()
        try:
            ex.load_markets()
            if "BTC/USDT" not in ex.markets:
                print(f"  x {ex.id} -- BTC/USDT not found")
                continue
            test = ex.fetch_ohlcv("BTC/USDT", "1h", limit=3, params=extra_params)
            if not test or len(test) == 0:
                print(f"  x {ex.id} -- fetch_ohlcv returned empty")
                continue
            print(f"  v Active exchange: {ex.id} (smoke-test OK: {len(test)} candles)")
            _active_exchange = ex
            _active_params   = extra_params
            return ex, extra_params
        except Exception as e:
            print(f"  x {ex.id} unavailable: {type(e).__name__}: {str(e)[:80]}")
    raise RuntimeError("No exchange accessible. Check Kaggle network connectivity.")

print("Searching for available exchange...")
_active_exchange, _active_params = _get_exchange()

# Diagnose available pairs
for sym in FORECASTER_SYMBOLS:
    available = sym in _active_exchange.markets
    print(f"  {sym}: {'OK' if available else 'NOT FOUND'}")

def fetch_candles(symbol: str, limit: int, timeframe: str = "1h") -> np.ndarray:
    """Returns array of shape (N, 6): [ts, open, high, low, close, volume]."""
    global _active_exchange, _active_params
    ex     = _active_exchange
    params = _active_params

    if symbol not in ex.markets:
        print(f"  ! {symbol} not found in {ex.id}")
        return np.empty((0, 6), dtype=np.float64)

    tf_ms      = TIMEFRAME_MS[timeframe]
    end_time   = int(time.time() * 1000)
    start_time = end_time - limit * tf_ms
    all_rows   = []
    since      = start_time

    while since < end_time:
        try:
            raw = ex.fetch_ohlcv(symbol, timeframe, since=since,
                                  limit=MAX_PER_REQUEST, params=params)
        except Exception as e:
            print(f"  ! Error on {ex.id} ({type(e).__name__}), reconnecting...")
            ex, params = _get_exchange()
            continue

        if not raw:
            break

        filtered = [row for row in raw if row[0] <= end_time]
        all_rows.extend(filtered)

        # Advance cursor by timestamp, NOT by batch size
        next_since = raw[-1][0] + tf_ms
        if next_since <= since:
            break
        since = next_since

        if len(all_rows) >= limit:
            break

        time.sleep(ex.rateLimit / 1000)

    if len(all_rows) == 0:
        print(f"  ! WARNING: 0 candles returned for {symbol}!")
        return np.empty((0, 6), dtype=np.float64)

    arr = np.array(all_rows, dtype=np.float64)
    if arr.ndim == 1:
        arr = arr.reshape(1, -1)

    _, idx = np.unique(arr[:, 0], return_index=True)
    arr = arr[idx]

    print(f"  -> {symbol}: {len(arr)} candles downloaded (requested: {limit})")
    return arr[-limit:]

print("Fetch function ready.")


In [ ]:
# Feature builder (16 features per timestep) + SOFT direction targets
#
# Features 0-9:  original (price, volume, RSI, MACD, BB, EMAs, etc.)
# Features 10-15: wavelet dual-path
#   10: denoised_trend / close - 1     (how far price is from its trend)
#   11: medium_band / close            (medium-freq oscillation, normalized)
#   12: high_band / close              (high-freq noise level, normalized)
#   13: wavelet_trend_slope            (trend direction & strength)
#   14: noise_to_signal_ratio          (clipped noise/signal energy ratio)
#   15: spectral_entropy               (frequency complexity, 0=pure tone, 1=noise)
#
# Labels: soft sigmoid transform of future return instead of hard 0/1.

def soft_label(future_return: float, scale: float = SOFT_LABEL_SCALE,
               clip_lo: float = SOFT_LABEL_CLIP[0],
               clip_hi: float = SOFT_LABEL_CLIP[1]) -> float:
    """Convert an EXCESS return (return minus per-series drift) into a soft probability via sigmoid.
    excess=0     → 0.50 (maximally uncertain)
    excess=+0.5% → ~0.68 (moderately bullish, scale=150)
    excess=+2.0% → ~0.95 (very bullish, scale=150)
    excess=-1.0% → ~0.18 (very bearish, scale=150)
    """
    sig = 1.0 / (1.0 + np.exp(-future_return * scale))
    return float(np.clip(sig, clip_lo, clip_hi))


def build_features_for_series(candles: np.ndarray, window_size: int = 128) -> tuple:
    closes  = candles[:, 4]
    opens   = candles[:, 1]
    highs   = candles[:, 2]
    lows    = candles[:, 3]
    volumes = candles[:, 5]

    # ── Original indicators (computed globally, fully causal) ──────────
    ema9_all   = ema(closes, 9)
    ema21_all  = ema(closes, 21)
    rsi14_all  = rsi(closes, 14)
    macd_line, _ = macd(closes, 12, 26, 9)
    bb_upper, bb_lower = bollinger(closes, 20, 2.0)
    sp_entropy = spectral_entropy_rolling(closes, window=32)

    X_list, y_list = [], []
    n = len(candles)
    mean_bar_return = float(np.mean(np.diff(closes) / closes[:-1]))  # per-series drift

    for i in range(window_size, n - HORIZON):
        w_start = i - window_size
        w_end   = i

        ref_close  = closes[w_start]
        max_volume = np.max(volumes[w_start:w_end]) if np.max(volumes[w_start:w_end]) > 0 else 1.0
        if max_volume == 0:
            max_volume = 1.0

        # ── Wavelet decomposition (computed LOCALLY per window to avoid data leakage) ──────────
        window_closes = closes[w_start:w_end]
        wv = wavelet_decompose_series(window_closes)
        wv_trend   = wv['trend']
        wv_medium  = wv['medium']
        wv_high    = wv['high']
        wv_slope   = wavelet_trend_slope(wv_trend, window=10)
        wv_med_nrg = wavelet_energy_ratio(wv_medium, window=20)
        wv_hi_nrg  = wavelet_energy_ratio(wv_high, window=20)

        # Normalize wavelet slope within the window for stability
        slope_std = np.std(wv_slope) + 1e-10

        rows = []
        for j in range(w_start, w_end):
            local_j = j - w_start
            c = closes[j]
            prev_ret = 0.0 if j == w_start else (c - closes[j-1]) / closes[j-1]
            r9   = ema9_all[j]   if not np.isnan(ema9_all[j])   else c
            r21  = ema21_all[j]  if not np.isnan(ema21_all[j])  else c
            rsi_v = rsi14_all[j] if not np.isnan(rsi14_all[j])  else 50.0
            macd_v = macd_line[j] if not np.isnan(macd_line[j]) else 0.0
            bbu  = bb_upper[j]   if not np.isnan(bb_upper[j])   else c
            bbl  = bb_lower[j]   if not np.isnan(bb_lower[j])   else c

            # Wavelet features (safe division & clipping for stability)
            med_e = wv_med_nrg[local_j] if wv_med_nrg[local_j] > 0 else 1e-6
            hi_e  = wv_hi_nrg[local_j]  if wv_hi_nrg[local_j] > 0  else 0.0
            noise_to_signal = float(np.clip(hi_e / med_e, 0.0, 10.0))

            rows.append([
                # ── Original 10 features ─────────────────────────────
                c / ref_close - 1,               # 0: normalized price
                volumes[j] / max_volume,         # 1: normalized volume
                rsi_v / 100.0,                   # 2: RSI [0,1]
                macd_v / c,                      # 3: MACD normalized
                prev_ret,                        # 4: previous return
                (highs[j] - lows[j]) / c,        # 5: range/close
                (r9 - c) / c,                    # 6: EMA9 distance
                (r21 - c) / c,                   # 7: EMA21 distance
                (bbu - c) / c,                   # 8: BB upper distance
                (bbl - c) / c,                   # 9: BB lower distance
                # ── Wavelet dual-path features (NEW) ─────────────────
                (wv_trend[local_j] - c) / c,           # 10: trend deviation
                wv_medium[local_j] / c,                # 11: medium-freq oscillation
                wv_high[local_j] / c,                  # 12: high-freq noise
                wv_slope[local_j] / slope_std,         # 13: trend slope (z-scored)
                noise_to_signal,                       # 14: noise/signal energy ratio (CLIPPED)
                sp_entropy[j],                         # 15: spectral entropy [0,1]
            ])

        X_list.append(rows)

        # ── SOFT LABELS: sigmoid transform of future return ──────────
        anchor_close = closes[i]
        future = []
        for h in range(1, HORIZON + 1):
            future_return = (closes[i + h] - anchor_close) / anchor_close
            excess_return = future_return - mean_bar_return * h  # de-mean drift -> 0.5 = no edge
            future.append(soft_label(excess_return))
        y_list.append(future)

    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)


def compute_volatility(candles: np.ndarray) -> np.ndarray:
    closes = candles[:, 4]
    n = len(closes)
    vol = np.zeros(n)
    for i in range(1, n):
        start = max(1, i - VOLATILITY_WINDOW)
        rets = (closes[start:i+1] - closes[start-1:i]) / closes[start-1:i]
        vol[i] = np.std(rets)
    return vol

print(f"Feature builder OK: {NUM_FEATURES} features, soft labels (scale={SOFT_LABEL_SCALE}).")
print(f"  Soft label examples:  +0.5% → {soft_label(0.005):.3f},  -0.5% → {soft_label(-0.005):.3f},  0.0% → {soft_label(0.0):.3f}")

In [ ]:
# ── Purged Embargoed Split ───────────────────────────────────────────────────

def purged_embargoed_split(X, y, horizon, embargo_frac, val_frac=0.1):
    total = len(X)
    val_size   = int(total * val_frac)
    val_start  = total - val_size
    embargo_sz = int(total * embargo_frac)
    train_end  = max(0, val_start - horizon - embargo_sz)
    purged = val_start - train_end
    return (
        X[:train_end], y[:train_end],
        X[val_start:],  y[val_start:],
        purged
    )

print("Split OK.")


In [ ]:
# ── Download and preprocessing ───────────────────────────────────────────────

total_needed = HISTORICAL_CANDLES + BACKTEST_HOLDOUT

all_X_train, all_y_train = [], []
all_X_val,   all_y_val   = [], []
agent_candles = None

for sym in FORECASTER_SYMBOLS:
    print(f"\nDownloading {total_needed} candles for {sym}...")
    raw = fetch_candles(sym, total_needed, TIMEFRAME)
    train_raw = raw[:-BACKTEST_HOLDOUT] if len(raw) > BACKTEST_HOLDOUT else raw

    print(f"  {len(train_raw)} candles for training")

    min_needed = WINDOW_SIZE + HORIZON + 1
    if len(train_raw) < min_needed:
        print(f"  ! Insufficient data (minimum: {min_needed}) -- skipping.")
        continue

    if sym == AGENT_SYMBOL:
        agent_candles = train_raw.copy()

    X, y = build_features_for_series(train_raw, WINDOW_SIZE)
    Xtr, ytr, Xv, yv, purged = purged_embargoed_split(
        X, y, HORIZON, EMBARGO_FRACTION, VALIDATION_FRACTION
    )
    all_X_train.append(Xtr);  all_y_train.append(ytr)
    all_X_val.append(Xv);     all_y_val.append(yv)
    print(f"  train={len(Xtr)}, val={len(Xv)}, purged={purged}")

if not all_X_train:
    raise RuntimeError("No symbol returned data.")

if agent_candles is None:
    raise RuntimeError(f"No data for agent ({AGENT_SYMBOL}).")

X_train = np.concatenate(all_X_train, axis=0)
y_train = np.concatenate(all_y_train, axis=0)
X_val   = np.concatenate(all_X_val,   axis=0)
y_val   = np.concatenate(all_y_val,   axis=0)

print(f"\nTotal dataset -- train: {X_train.shape}, val: {X_val.shape}")


In [ ]:
# ── Pre-compute agent features and volatility ────────────────────────────────

print("Pre-computing agent features...")
agent_X, _ = build_features_for_series(agent_candles, WINDOW_SIZE)
agent_vol  = compute_volatility(agent_candles)
agent_vol_samples = agent_vol[WINDOW_SIZE : len(agent_candles) - HORIZON]

print(f"Agent features: {agent_X.shape}, volatility: {agent_vol_samples.shape}")


In [ ]:
# Save the dataset bundle. Single .npz with everything the training notebooks need.
# Publish the resulting file as the Kaggle Dataset acaurangel/ai-spot-trading-dataset.

OUT = f'{WORK_DIR}/dataset.npz'
np.savez_compressed(
    OUT,
    X_train=X_train, y_train=y_train,
    X_val=X_val, y_val=y_val,
    agent_X=agent_X, agent_candles=agent_candles, agent_vol_samples=agent_vol_samples,
)
print(f'Saved {OUT} | size: {os.path.getsize(OUT)/1e6:.1f} MB')
print()
print('Shapes:')
print(f'  X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'  X_val:   {X_val.shape},  y_val:   {y_val.shape}')
print(f'  agent_X: {agent_X.shape}, agent_candles: {agent_candles.shape}, agent_vol: {agent_vol_samples.shape}')
print()
print('Next: Notebook output panel -> Create new dataset -> slug acaurangel/ai-spot-trading-dataset')